In [ ]:
# ==========================================
# CELL 1: Install Libraries & Import
# (Run this first. It will download the Hugging Face tools to Colab)
# ==========================================
!pip install transformers datasets accelerate

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import torch

# Hugging Face specific imports
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer


In [ ]:
# ==========================================
# CELL 2: Load Data & Convert to Hugging Face Format
# ==========================================
print("Loading dataset...")
# Note: Since you dragged and dropped the file into Colab, it is usually
# in the main folder, so we just use the filename directly.
df = pd.read_csv('sentiment_dataset.csv')
df['cleaned_text'] = df['cleaned_text'].fillna('')

# Remove rows where 'label' is NaN, as train_test_split with stratify cannot handle them.
df.dropna(subset=['label'], inplace=True)
# Ensure 'label' column is of integer type after dropping NaNs
df['label'] = df['label'].astype(int)

# Split using pandas/sklearn just like before
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Hugging Face uses its own super-fast 'Dataset' object instead of Pandas DataFrames.
# We convert our pandas dataframes into Hugging Face datasets here:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Group them into a single dictionary
dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("Dataset successfully converted!")

Loading dataset...
Dataset successfully converted!


In [ ]:
# ==========================================
# CELL 3: The BERT Tokenizer & The 512 Limit
# ==========================================
print("Downloading BERT Tokenizer...")
# This automatically downloads Google's official vocabulary rules for BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    # This replaces Keras pad_sequences.
    # truncation=True rigorously enforces the 512 word/token limit.
    # padding='max_length' adds zeros to short articles.
    return tokenizer(
        examples["cleaned_text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

print("Tokenizing the dataset (this takes about 1-2 minutes)...")
# map() applies our tokenizer to every article simultaneously
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Tokenizing the dataset (this takes about 1-2 minutes)...


Map:   0%|          | 0/35911 [00:00<?, ? examples/s]

Map:   0%|          | 0/8978 [00:00<?, ? examples/s]

In [ ]:
# ==========================================
# CELL 4: Define the Evaluation Metrics
# ==========================================
# Hugging Face needs us to explicitly tell it how to grade the exam.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Logits are the raw math scores; we use argmax to turn them into 0 or 1
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions)

    return {"accuracy": acc, "f1": f1}

In [ ]:
# ==========================================
# CELL 5: Download BERT & Train!
# ==========================================
print("Downloading BERT Model weights (110 million parameters)...")
# num_labels=2 tells BERT we are doing Binary Classification (Fake vs Real)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Set up the rules for training
training_args = TrainingArguments(
    output_dir="./bert_fake_news_model",
    eval_strategy="epoch",      # Give us a grade at the end of every epoch
    learning_rate=2e-5,         # Extremely tiny learning rate so we don't cause "Catastrophic Forgetting"
    per_device_train_batch_size=8, # Reduced batch size to prevent OOM errors
    per_device_eval_batch_size=8,  # Reduced batch size to prevent OOM errors
    num_train_epochs=3,         # 3 epochs is the industry standard for fine-tuning BERT
    weight_decay=0.01,
)

# The Trainer is Hugging Face's automated training engine
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

print("\n--- Final Evaluation ---")
trainer.evaluate()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.000066,0.002351,0.999332,0.999299
2,0.002091,0.003820,0.999332,0.999300
3,0.002113,0.000308,0.999889,0.999883


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Final Evaluation ---


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.002113,0.000308,3,0.999889,0.999883


{'eval_loss': 0.00030752946622669697,
 'eval_accuracy': 0.9998886166184006,
 'eval_f1': 0.9998832457676591}

In [ ]:
# ==========================================
# CELL 6: Save Final Model and Zip for Download
# ==========================================
print("Saving the final model and tokenizer...")
# 1. Save the final model weights and blueprint
trainer.save_model("./final_fake_news_bert")
# 2. Save the tokenizer so you can process future text exactly the same way
tokenizer.save_pretrained("./final_fake_news_bert")

print("Compressing the model into a zip file...")
# 3. Zip the folder so it's easy to download
import shutil
shutil.make_archive("final_fake_news_bert", 'zip', "./final_fake_news_bert")

print("Zipping complete! Triggering download...")
# 4. Automatically prompt your browser to download the file
from google.colab import files
files.download("final_fake_news_bert.zip")

Saving the final model and tokenizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Compressing the model into a zip file...
Zipping complete! Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After running the above cell, you should see if any checkpoints (e.g., `checkpoint-500`, `checkpoint-1000`) or a `model` directory exist. If the training completed all 3 epochs, you might see a `checkpoint-XXXX` where `XXXX` is the total number of training steps, or a final model saved. If not, you can simply re-run Cell 5 (`eZBdmgoTDAAt`) to attempt to resume training.